# 02｜樣本定稿：從審閱表到 `events.csv`（v6）

上一本 notebook（`build_sample_v5`）從 TEJ 的重大訊息挑出 250 宗候選交易。之後每一宗都對照公告原文審閱過，結果記在 `review_final.xlsx`。
這一本做最後一步：**把審閱結果變成可以直接拿去跑事件研究的事件表**。

**流程（共 4 格，照順序一格一格按）**
1. 連接 Google Drive，確認要用的檔案都在
2. 檢查審閱表有沒有漏填，並統計「250 宗候選是怎麼變成最後樣本的」
3. 用 TEJ 的資料計算每一宗的交易金額（外幣換算成新台幣）和相對規模，並核對公告當天是不是上市櫃
4. 套用最後兩個機械式的條件（相對規模 ≥ 1%、公告當天是上市櫃），輸出 `events.csv`

**用到的檔案**（都在 Drive 的 `ma_event_study/`）

| 檔案 | 位置 | 內容 |
|---|---|---|
| `review_final.xlsx` | `work/` | 審閱結果（250 宗＋C 抽查 60 宗） |
| `tej_shares_2019～2023.xlsx` | `data/` | TEJ 未調整股價(日)：收盤價、流通在外股數、市值、市場別 |
| `tej_fx_monthly.xlsx` | `data/` | TEJ 總經資料：月底匯率 |

**兩個要記得的定義**（面試會被問）
- **相對規模** ＝ 交易金額 ÷ 主併公司在「公告前一個月底」的市值。用公告前的市值，是因為公告後股價已經反應了這宗交易。發行新股的交易沒有金額，用「新股數 ÷ 公告前一個月底的流通在外股數」，意思相同。
- **外幣金額** 用「公告前一個月底」的匯率換算，和市值是同一個時點，而且都是公告前就已知的數字。

## 第 1 格｜連接 Drive，確認檔案都在

這一格不做任何計算，只是：
- 連接你的 Google Drive（會跳出視窗要你按「允許」）
- 列出需要的 7 個檔案，在不在都會顯示
- 把 TEJ 檔案的欄位名稱和前 3 列印出來，確認程式讀到的格式正確

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re
import numpy as np
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 70); pd.set_option('display.max_rows', 300)

BASE = '/content/drive/MyDrive/ma_event_study'
DATA_DIR, WORK_DIR = f'{BASE}/data', f'{BASE}/work'
REVIEW_FILE = f'{WORK_DIR}/review_final.xlsx'

# ── 需要的檔案在不在 ──
data_files = sorted(os.listdir(DATA_DIR))
share_files = [f for f in data_files if 'tej_shares' in f.lower() and f.lower().endswith('.xlsx')]
fx_files    = [f for f in data_files if 'tej_fx' in f.lower() and f.lower().endswith('.xlsx')]
print('審閱表          ：', '✓ 找到' if os.path.exists(REVIEW_FILE) else '✗ 找不到 review_final.xlsx，請確認已上傳到 work 資料夾')
print(f'TEJ 股數市值檔  ： 找到 {len(share_files)} 個（應該是 5 個）', share_files)
print(f'TEJ 匯率檔      ： 找到 {len(fx_files)} 個（應該是 1 個）', fx_files)

def read_xlsx(path, **kw):
    """讀 Excel。全部先當成文字讀進來，之後再自己轉數字，避免代號 0050 被讀成 50"""
    return pd.read_excel(path, dtype=str, **kw)

# ── 各看一眼：欄位名稱和前 3 列 ──
if share_files:
    t = read_xlsx(f'{DATA_DIR}/{share_files[0]}', nrows=5)
    print('\n【股數市值檔】', share_files[0]); print('欄位：', t.columns.tolist()); display(t.head(3))
if fx_files:
    t = read_xlsx(f'{DATA_DIR}/{fx_files[0]}', nrows=5)
    print('\n【匯率檔】', fx_files[0]); print('欄位：', t.columns.tolist()); display(t.head(3))

## 第 2 格｜檢查審閱表，統計樣本是怎麼篩出來的

**檢查**（任何一項沒過，程式會停下來並告訴你是哪幾宗）
- 250 宗都標了「已核對」
- 沒有「待查」
- 納入的每一宗，五個特徵欄位都有填，而且有金額、新股數或原幣金額其中之一
- 排除的每一宗都有排除原因

**統計**：印出「候選 250 宗 → 各種原因排除幾宗 → 剩幾宗」的表。這張表會放進 GitHub 和方法說明，讓讀的人知道樣本不是隨便挑的。

In [ ]:
def s(v):
    """把儲存格的內容變成乾淨的文字：空白回傳 ''，1.0 變成 '1'"""
    if v is None or (isinstance(v, float) and np.isnan(v)): return ''
    if isinstance(v, (int, float)) and float(v) == int(v): return str(int(v))
    return str(v).strip()

def load_sheet(name):
    df = pd.read_excel(REVIEW_FILE, sheet_name=name, dtype=object)
    df = df[df['編號'].notna()].copy()
    df['編號'] = df['編號'].map(s)
    return df.set_index('編號')

rv, audit = load_sheet('審閱'), load_sheet('C抽查')
FX_COL = [c for c in rv.columns if c.startswith('原幣金額')][0]
FEATURES = ['標的公司', '支付方式', '跨國', '產業關係', '標的上市櫃']
for c in ['納入', '排除原因', '已核對', '交易金額_百萬元', '發行新股_千股', '首次公告日_更正', '備註', '關係人', '盤中盤後', FX_COL] + FEATURES:
    rv[c] = rv[c].map(s)
inc = rv[rv['納入'] == '1']

# ── 檢查 ──
problems = {
    '「已核對」沒有填 1':            rv.index[rv['已核對'] != '1'].tolist(),
    '「納入」不是 1 或 0':            rv.index[~rv['納入'].isin(['1', '0'])].tolist(),
    '排除但沒有排除原因':             rv.index[(rv['納入'] == '0') & (rv['排除原因'] == '')].tolist(),
    '納入但填了排除原因':             inc.index[inc['排除原因'] != ''].tolist(),
    '納入但特徵欄位空白或待查':       inc.index[inc[FEATURES].isin(['', '待查']).any(axis=1)].tolist(),
    '支付方式不是 cash/stock/mixed':  inc.index[~inc['支付方式'].isin(['cash', 'stock', 'mixed'])].tolist(),
    '產業關係不是 同業/上下游/無關':  inc.index[~inc['產業關係'].isin(['同業', '上下游', '無關'])].tolist(),
    '納入但金額、新股數、原幣金額都空白': inc.index[(inc['交易金額_百萬元'] == '') & (inc['發行新股_千股'] == '') & (inc[FX_COL] == '')].tolist(),
    'C 抽查沒有核對完':               audit.index[audit['已核對'].map(s) != '1'].tolist(),
}
print('【檢查審閱表】')
for k, v in problems.items():
    print(f"  {'✓' if not v else '✗'} {k}" + ('' if not v else f'：編號 {v}'))
assert not any(problems.values()), '審閱表還有沒填好的地方（上面打 ✗ 的），請先到 review_final.xlsx 補好，再重新執行這一格'

# ── 樣本是怎麼篩出來的 ──
ORDER = ['被併方', '興櫃', '非首次公告', '少數股權', '集團內部', '資產購買', '規模小於1%', '60日內重複', '其他']
MEANING = {'被併方': '本公司是被收購的一方', '興櫃': '公告當時不是上市櫃公司', '非首次公告': '同一宗交易更早已公告過',
           '少數股權': '交易後持股沒有超過 50%', '集團內部': '交易前已控制標的，或向母公司、兄弟公司購買', '資產購買': '買的是土地廠房，不是事業',
           '規模小於1%': '交易金額不到主併公司市值的 1%', '60日內重複': '同公司 60 個交易日內的第二宗', '其他': '無償受讓、事後補公告、金額或持股比例查不到'}
cnt = rv.loc[rv['納入'] == '0', '排除原因'].value_counts()
assert set(cnt.index) <= set(ORDER), f'有不認得的排除原因：{set(cnt.index) - set(ORDER)}'
funnel = [('逐宗審閱的候選交易', len(rv), '')]
for r in ORDER:
    funnel.append((f'  排除：{r}', -int(cnt.get(r, 0)), MEANING[r]))
funnel.append(('審閱後納入', len(inc), ''))
funnel = pd.DataFrame(funnel, columns=['步驟', '宗數', '說明'])
print(f'\n【樣本篩選過程】'); display(funnel)
n_miss = int((audit['應該納入'].map(s) == '1').sum())
print(f'C 抽查：程式建議排除的交易裡隨機抽 {len(audit)} 宗，逐宗看過後應該納入的有 {n_miss} 宗')

## 第 3 格｜計算交易金額和相對規模，核對上市櫃

這一格把 TEJ 的資料接上來，對納入的每一宗算出：

| 欄位 | 怎麼算 |
|---|---|
| 公告前一個月底的市值、股數 | 從 `tej_shares` 找公告那個月的前一個月，最後一個有資料的交易日 |
| 交易金額（新台幣百萬元） | 新台幣的直接用；外幣用公告前一個月底的匯率換算 |
| 相對規模 | 現金：金額 ÷ 市值。發行新股：新股數 ÷ 流通在外股數。兩種都有（mixed）就相加 |
| 公告當天的市場別 | 從 `tej_shares` 看公告日（或之前最近一個交易日）是上市、上櫃還是興櫃 |

**人民幣和泰銖**：TEJ 沒有直接的「兌新台幣」匯率，所以用美元轉一手：人民幣兌新台幣 ＝（美元兌新台幣）÷（美元兌人民幣）。

**事件日 t0**：沿用上一本 notebook 算好的（公告當天；盤後或非交易日公告的順延到下一個交易日）。只有改過首次公告日的那一宗（編號 195 環球晶，11/30 開盤前公告）在這裡重新指定。

最後印出四個檢查：市值和審閱表上的數字差多少、哪幾宗相對規模不到 1%、哪幾宗公告當天不是上市櫃、同一家公司有沒有靠太近的兩宗。

In [ ]:
# ══ 1) 讀 TEJ 股數市值檔（5 個年度檔合併）══
KEY = {'代號': 'stock_id', '年月日': 'date', '收盤價': 'close', '流通在外股數': 'shares_k', '市值': 'mktcap', '市場別': 'market'}
parts = []
for f in share_files:
    df = read_xlsx(f'{DATA_DIR}/{f}')
    df.columns = [str(c).strip() for c in df.columns]
    ren = {c: v for c in df.columns for k, v in KEY.items() if k in c}        # 用關鍵字對應欄位，不怕括號是全形或半形
    df = df.rename(columns=ren)
    miss = set(KEY.values()) - set(df.columns)
    assert not miss, f'{f} 找不到欄位 {miss}，實際欄位是 {df.columns.tolist()}'
    parts.append(df[list(KEY.values())]); print(f'  {f}: {len(df):,} 筆')
px = pd.concat(parts, ignore_index=True)
px['stock_id'] = px['stock_id'].astype(str).str.strip().str.split().str[0].str.replace(r'\.0$', '', regex=True)
px['date'] = pd.to_datetime(px['date'], errors='coerce')
for c in ['close', 'shares_k', 'mktcap']:
    px[c] = pd.to_numeric(px[c].astype(str).str.replace(',', ''), errors='coerce')
px = px.dropna(subset=['date']).drop_duplicates(['stock_id', 'date']).sort_values(['stock_id', 'date'])
print(f'股數市值資料：{len(px):,} 筆，{px["stock_id"].nunique()} 家公司，{px["date"].min().date()} ~ {px["date"].max().date()}')
print('市場別的寫法和筆數：', px['market'].value_counts(dropna=False).to_dict())

def market_name(v):
    """把 TEJ 的市場別統一成 上市／上櫃／興櫃"""
    v = str(v).upper()
    if 'ROTC' in v or '興' in v or v in ('REG', 'ESB'): return '興櫃'
    if 'TSE' in v or 'TWSE' in v or '上市' in v:         return '上市'
    if 'OTC' in v or 'TPEX' in v or '上櫃' in v:         return '上櫃'
    return f'不認得（{v}）'

day_count = px.groupby('date')['stock_id'].size()
calendar = pd.DatetimeIndex(day_count[day_count >= 20].index)                 # 交易日：至少 20 家公司有資料的日子
by_firm = {k: g.set_index('date') for k, g in px.groupby('stock_id')}

# ══ 2) 讀匯率檔 ══
fx = read_xlsx(f'{DATA_DIR}/{fx_files[0]}')
fx.columns = [str(c).strip() for c in fx.columns]
fx['數值'] = pd.to_numeric(fx['數值'], errors='coerce')
fx['年月'] = fx['年月'].astype(str).str.replace('-', '/').str[:7]
FX_CODE = {'USD': 'MC02', 'EUR': 'MC0402', 'GBP': 'MC03', 'SGD': 'MC10', 'JPY': 'MC05'}   # 月底「兌新台幣」匯率的 TEJ 代號
CROSS   = {'CNY': 'CK13', 'THB': 'CT01'}                                                  # 只有「美元兌該貨幣」，要用美元轉一手
series = {c: fx[fx['代號'] == c].set_index('年月')['數值'] for c in list(FX_CODE.values()) + list(CROSS.values())}
print('\n用到的匯率項目：')
for c in series: print(f'  {c:7s}', fx.loc[fx['代號'] == c, '名稱'].iloc[0], f'（{len(series[c])} 個月）')

def fx_rate(cur, ym):
    """ym 這個月的月底，1 單位外幣換多少新台幣"""
    if cur in FX_CODE: return series[FX_CODE[cur]][ym]
    return series['MC02'][ym] / series[CROSS[cur]][ym]

# ══ 3) 逐宗計算 ══
T0_OVERRIDE = {'195': ('2020-11-30', '盤前')}      # 改過首次公告日的：環球晶 2020/11/30 開盤前發重大訊息 → 事件日就是 11/30

rows = []
for no, r in inc.iterrows():
    sid = s(r['代號'])
    ann = pd.to_datetime(r['首次公告日_更正'] if r['首次公告日_更正'] else r['首次公告日'])
    if r['首次公告日_更正']:
        assert no in T0_OVERRIDE, f'編號 {no} 改過首次公告日，但程式裡沒有指定它的事件日，請告訴 Claude'
        t0, timing = pd.Timestamp(T0_OVERRIDE[no][0]), T0_OVERRIDE[no][1]
        assert t0 in calendar, f'編號 {no} 指定的事件日 {t0.date()} 不是交易日'
    else:
        t0, timing = pd.to_datetime(r['事件日t0']), r['盤中盤後']
    g = by_firm.get(sid)
    if g is None:
        rows.append(dict(編號=no, 問題='tej_shares 裡沒有這家公司')); continue
    month_start = ann.replace(day=1)
    prev = g[(g.index < month_start) & (g.index >= month_start - pd.Timedelta(days=20))].dropna(subset=['mktcap'])   # 前一個月底（往前最多找 20 天）
    on_day = g[g.index <= ann]
    pm = prev.iloc[-1] if len(prev) else None
    ym_prev = (ann.to_period('M') - 1).strftime('%Y/%m')

    # 交易金額：新台幣直接用；外幣（例如 "USD 403 + CNY 345.076"）逐項換算後相加
    amt, fx_note = (float(r['交易金額_百萬元']) if r['交易金額_百萬元'] else 0.0), ''
    if r[FX_COL]:
        for part in r[FX_COL].split('+'):
            cur, val = part.split()
            rate = fx_rate(cur, ym_prev)
            amt += float(val) * rate
            fx_note += f'{cur} {float(val):g} × {rate:.4f}；'
    new_k = float(r['發行新股_千股']) if r['發行新股_千股'] else 0.0
    if r['支付方式'] == 'cash':    new_k = 0.0
    if r['支付方式'] == 'stock' and new_k > 0: amt = 0.0          # 發行新股的交易看股數；表上另外記的金額只是參考，不重複計算

    mktcap = pm['mktcap'] if pm is not None else np.nan
    shares = pm['shares_k'] if pm is not None else np.nan
    stock_value = new_k / shares * mktcap if new_k > 0 else 0.0     # 新股 ÷ 原有股數，再乘上市值，換成金額
    deal_value = amt + stock_value
    rows.append(dict(編號=no, stock_id=sid, 公司=r['公司'], 公告日=ann, 時點=timing, t0=t0, 支付=r['支付方式'],
                     前月底=prev.index[-1] if len(prev) else pd.NaT, 市值_前月底=mktcap, 股數_千股=shares,
                     現金或折算金額=round(amt, 1), 新股_千股=new_k, 交易金額_百萬=round(deal_value, 1),
                     相對規模=deal_value / mktcap if mktcap else np.nan,
                     市場別=market_name(on_day['market'].iloc[-1]) if len(on_day) else '查不到',
                     審閱表市值=pd.to_numeric(r['市值_百萬元'], errors='coerce'), 匯率=fx_note))
calc = pd.DataFrame(rows).set_index('編號')
assert '問題' not in calc.columns, f"有公司在 tej_shares 裡找不到：{calc.index[calc['問題'].notna()].tolist()}"

# ══ 4) 四個檢查 ══
assert calc['相對規模'].notna().all(), f"這幾宗算不出相對規模（前一個月底沒有市值資料）：{calc.index[calc['相對規模'].isna()].tolist()}，請截圖給 Claude"
calc['市值差%'] = (calc['市值_前月底'] / calc['審閱表市值'] - 1) * 100
print(f'\n【檢查 1】市值：和審閱表上的數字（上一本 notebook 用另一份 TEJ 檔案算的）差超過 2% 的有 {(calc["市值差%"].abs() > 2).sum()} 宗')
display(calc.loc[calc['市值差%'].abs() > 2, ['公司', '公告日', '前月底', '市值_前月底', '審閱表市值', '市值差%']].round({'市值差%': 1}))

print('【檢查 2】相對規模最小的 12 宗（不到 1% 的在下一格排除）')
display(calc.sort_values('相對規模')[['公司', '公告日', '支付', '交易金額_百萬', '市值_前月底', '新股_千股', '股數_千股', '相對規模', '匯率']].head(12)
        .assign(相對規模=lambda d: (d['相對規模'] * 100).round(2).astype(str) + '%'))

print('【檢查 3】公告當天的市場別：', calc['市場別'].value_counts().to_dict())
display(calc.loc[~calc['市場別'].isin(['上市', '上櫃']), ['公司', '公告日', '市場別']])

print('【檢查 4】同一家公司 60 個交易日內有兩宗的（應該沒有）')
pos = pd.Series(calendar.searchsorted(calc['t0']), index=calc.index)
tmp = calc.assign(pos=pos).sort_values(['stock_id', 'pos']); tmp['距上一宗_交易日'] = tmp.groupby('stock_id')['pos'].diff()
display(tmp.loc[tmp['距上一宗_交易日'] < 60, ['公司', '公告日', '距上一宗_交易日']])

print(f'\n外幣交易 {int((calc["匯率"] != "").sum())} 宗、發行新股 {int((calc["新股_千股"] > 0).sum())} 宗、沒有算出相對規模的 {int(calc["相對規模"].isna().sum())} 宗')

## 第 4 格｜輸出 `events.csv`

1. 套用最後兩個條件：相對規模不到 1% 的、公告當天不是上市櫃的，在這裡排除（這兩個條件要有 TEJ 的數字才能確定，所以放在最後）
2. 加上之後分析會用到的標記
   - `related_party`：公告的「關係人」欄填「是」
   - `ky_firm`：公司名稱有「-KY」（註冊在海外、營運多在大陸或東南亞的公司）
   - `flag_...`：審閱時在備註標出的邊界個案（產業、持股、跨國、規模為下限、公告前已有報導）。主要結果照原本的編碼跑，穩健性檢驗時把這些拿掉或改編再跑一次，看結論會不會變
3. 存成兩個檔案
   - `work/events.csv`：完整版，含 TEJ 的市值和相對規模。**只放在 Drive，不要上傳 GitHub**（TEJ 是學校授權的資料）
   - `work/events_public.csv`：公開版，只有從公開公告整理的欄位，可以放 GitHub
4. 印出樣本的基本統計

In [ ]:
ev = calc.join(inc[['標的公司', '跨國', '產業關係', '標的上市櫃', '關係人', '主併產業', '備註']])

# ── 1) 最後兩個條件 ──
late = pd.Series('', index=ev.index)
late[~ev['市場別'].isin(['上市', '上櫃'])] = '興櫃'
late[(late == '') & ~(ev['相對規模'] >= 0.01)] = '規模小於1%'
print('【這一格排除的】'); display(ev.loc[late != '', ['公司', '公告日', '交易金額_百萬', '市值_前月底', '相對規模', '市場別']].assign(原因=late[late != '']))
ev = ev[late == ''].copy()

funnel2 = funnel.copy()
funnel2.loc[len(funnel2)] = ('  排除：規模小於1%（用 TEJ 股數與匯率計算後）', -int((late == '規模小於1%').sum()), '')
funnel2.loc[len(funnel2)] = ('  排除：興櫃（用 TEJ 市場別核對後）', -int((late == '興櫃').sum()), '')
funnel2.loc[len(funnel2)] = ('最後樣本', len(ev), '')
print('【樣本篩選過程（完整）】'); display(funnel2)

# ── 2) 整理欄位 ──
note = ev['備註']
out = pd.DataFrame({
    'event_id': ev.index, 'stock_id': ev['stock_id'], 'firm_name': ev['公司'].str.split().str[1:].str.join(' '),
    'announce_date': ev['公告日'].dt.date, 'announce_timing': ev['時點'], 't0': ev['t0'].dt.date, 'year': ev['公告日'].dt.year,
    'market': ev['市場別'], 'acq_industry': ev['主併產業'], 'target_name': ev['標的公司'],
    'payment': ev['支付'], 'cross_border': ev['跨國'].astype(int),
    'relatedness': ev['產業關係'].map({'同業': 'same_industry', '上下游': 'vertical', '無關': 'unrelated'}),
    'target_listed': ev['標的上市櫃'].astype(int),
    'deal_value_mn': ev['交易金額_百萬'], 'acq_mktcap_mn': ev['市值_前月底'].round(0), 'rel_size': ev['相對規模'].round(4),
    'related_party': (ev['關係人'] == '是').astype(int), 'ky_firm': ev['公司'].str.contains('-KY').astype(int),
    'flag_industry_border': note.str.contains('【產業邊界】').astype(int), 'flag_stake_border': note.str.contains('【持股邊界】').astype(int),
    'flag_crossborder_border': note.str.contains('【跨國邊界】').astype(int), 'flag_size_lower_bound': note.str.contains('【規模為下限】').astype(int),
    'flag_pre_leak': note.str.contains('【公告前已有報導】').astype(int), 'note': note,
}).sort_values(['announce_date', 'stock_id']).reset_index(drop=True)

PUBLIC = ['event_id', 'stock_id', 'firm_name', 'announce_date', 'announce_timing', 'year', 'target_name', 'payment', 'cross_border', 'relatedness',
          'target_listed', 'related_party', 'ky_firm', 'flag_industry_border', 'flag_stake_border', 'flag_crossborder_border', 'flag_size_lower_bound', 'flag_pre_leak']
out.to_csv(f'{WORK_DIR}/events.csv', index=False, encoding='utf-8-sig')
out[PUBLIC].to_csv(f'{WORK_DIR}/events_public.csv', index=False, encoding='utf-8-sig')
funnel2.to_csv(f'{WORK_DIR}/sample_funnel.csv', index=False, encoding='utf-8-sig')
print(f'已存檔：events.csv（{len(out)} 宗，{out["stock_id"].nunique()} 家公司）、events_public.csv、sample_funnel.csv')

# ── 3) 樣本的樣子 ──
print('\n【各年度宗數】', out['year'].value_counts().sort_index().to_dict())
print('【上市／上櫃】', out['market'].value_counts().to_dict())
print('【支付方式】', out['payment'].value_counts().to_dict())
print('【跨國】', out['cross_border'].map({1: '跨國', 0: '國內'}).value_counts().to_dict())
print('【產業關係】', out['relatedness'].value_counts().to_dict())
print('【標的是上市櫃公司】', int(out['target_listed'].sum()), '宗')
print('【標記】關係人', int(out['related_party'].sum()), '｜KY 公司', int(out['ky_firm'].sum()), '｜產業邊界', int(out['flag_industry_border'].sum()),
      '｜持股邊界', int(out['flag_stake_border'].sum()), '｜跨國邊界', int(out['flag_crossborder_border'].sum()),
      '｜規模為下限', int(out['flag_size_lower_bound'].sum()), '｜公告前已有報導', int(out['flag_pre_leak'].sum()))
print('\n【相對規模的分布】（1 = 交易金額等於主併公司市值）')
display(out['rel_size'].describe(percentiles=[.1, .25, .5, .75, .9]).round(3).to_frame().T)
print('【主併公司市值的分布】（百萬元）')
display(out['acq_mktcap_mn'].describe(percentiles=[.1, .25, .5, .75, .9]).round(0).to_frame().T)
display(out.drop(columns='note').head(10))

## 做完之後

- `events.csv` 是下一本 notebook（事件研究）的輸入。
- 第 2、4 格印出的「樣本篩選過程」和第 4 格的樣本統計，之後會放進 README 和一頁式摘要。
- **想改樣本怎麼辦**：不要直接改 `events.csv`。回到 `review_final.xlsx` 改，再把這一本從頭跑一次。這樣每一個數字都能追回到審閱表和公告原文。